#**Extracting Data from Pdf**

In [46]:
pip install PyPDF2

In [47]:
import PyPDF2

In [48]:
f=open("/content/Artificial intelligence.pdf","rb")

In [49]:
pdf_reader=PyPDF2.PdfReader(f)

In [50]:
len(pdf_reader.pages)

4

In [51]:
from PyPDF2 import PdfReader
reader = PdfReader("/content/Artificial intelligence.pdf")

# Extract multiple pages
doc = [0,1,2,3]   # 0-based indexing (page 2 & 3)

text = ""
for i in doc:
    text += reader.pages[i].extract_text()

print(text)

Artificial intelligence  (AI) is the capability of  computational systems  to perform tasks 
typically associated with  human intelligence , such as  learning , reasoning , problem -
solving , perception , and  decision -making . It is a  field of 
research  in engineering , mathematics  and computer science  that develops and studies 
methods and  software  that enable machines to perceive their environment and 
use learning  and intelligence  to take actions that maximize their chances of achieving 
defined goals.[1] 
High -profile  applications of AI  include advanced  web search engines , chatbots , virtual 
assistants , autonomous vehicles , and play and analysis in  strategy 
games  (e.g.,  chess  and Go). Since the 2020s,  generative AI  has become widely available to 
generate images, audio, and videos from text prompts.  
The traditional goals of AI research include learning,  reasoning , knowledge 
representation , planning , natural language processing , and  perception , as

In [52]:
f.close()

#**Data Preprocessing**

In [53]:
import nltk
import re
import string

In [54]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [55]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

# Remove URLs, citations [1], special characters
text = re.sub(r'http\S+', '', text)           # remove URLs
text = re.sub(r'\[\d+\]', '', text)           # remove citations
text = re.sub(r'[^a-zA-Z\s]', '', text)       # keep only alphabets
text = text.lower()                           # lowercase


# 3. Tokenization
tokens = word_tokenize(text)


# 4. Stopword Removal
stop_words = set(stopwords.words('english'))
tokens = [word for word in tokens if word not in stop_words]


# 5. Lemmatization
lemmatizer = WordNetLemmatizer()
tokens = [lemmatizer.lemmatize(word) for word in tokens]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [56]:
# 6. Final Clean Text
clean_text = " ".join(tokens)

print("\nCleaned Text Sample:\n", clean_text[:500])


Cleaned Text Sample:
 artificial intelligence ai capability computational system perform task typically associated human intelligence learning reasoning problem solving perception decision making field research engineering mathematics computer science develops study method software enable machine perceive environment use learning intelligence take action maximize chance achieving defined goal high profile application ai include advanced web search engine chatbots virtual assistant autonomous vehicle play analysis str


#**CBOW**

In [57]:
#Tokenization
words=clean_text.split()

In [58]:
#Vocabulary
vocab=list((words))
print(vocab)
vocab_size=len(vocab)

['artificial', 'intelligence', 'ai', 'capability', 'computational', 'system', 'perform', 'task', 'typically', 'associated', 'human', 'intelligence', 'learning', 'reasoning', 'problem', 'solving', 'perception', 'decision', 'making', 'field', 'research', 'engineering', 'mathematics', 'computer', 'science', 'develops', 'study', 'method', 'software', 'enable', 'machine', 'perceive', 'environment', 'use', 'learning', 'intelligence', 'take', 'action', 'maximize', 'chance', 'achieving', 'defined', 'goal', 'high', 'profile', 'application', 'ai', 'include', 'advanced', 'web', 'search', 'engine', 'chatbots', 'virtual', 'assistant', 'autonomous', 'vehicle', 'play', 'analysis', 'strategy', 'game', 'eg', 'chess', 'go', 'since', 'generative', 'ai', 'become', 'widely', 'available', 'generate', 'image', 'audio', 'video', 'text', 'prompt', 'traditional', 'goal', 'ai', 'research', 'include', 'learning', 'reasoning', 'knowledge', 'representation', 'planning', 'natural', 'language', 'processing', 'percept

In [59]:
#Word to index mapping
word_to_index={word: i for i,word in enumerate(vocab)}
index_to_word={i: word for word,i in word_to_index.items()}

In [60]:
##Generate Training Data (window size=1)
window_size=1
training_data=[]

for i in range(window_size,len(words)-window_size):
    context=[]
    for j in range(-window_size,window_size+1):
        if j!=0:
            context.append(word_to_index[words[i+j]])

    target=word_to_index[words[i]]
    training_data.append((context,target))
print(training_data)

[([766, 672], 223), ([223, 229], 672), ([672, 850], 229), ([229, 233], 850), ([850, 6], 233), ([233, 669], 6), ([6, 863], 669), ([669, 9], 863), ([863, 889], 9), ([9, 223], 889), ([889, 869], 223), ([223, 399], 869), ([869, 838], 399), ([399, 276], 838), ([838, 941], 276), ([276, 658], 941), ([941, 543], 658), ([658, 928], 543), ([543, 297], 928), ([928, 325], 297), ([297, 22], 325), ([325, 922], 22), ([22, 24], 922), ([922, 25], 24), ([24, 664], 25), ([25, 263], 664), ([664, 28], 263), ([263, 29], 28), ([28, 903], 29), ([29, 31], 903), ([903, 32], 31), ([31, 906], 32), ([32, 869], 906), ([906, 223], 869), ([869, 608], 223), ([223, 633], 608), ([608, 38], 633), ([633, 39], 38), ([38, 40], 39), ([39, 41], 40), ([40, 475], 41), ([41, 43], 475), ([475, 44], 43), ([43, 901], 44), ([44, 672], 901), ([901, 858], 672), ([672, 48], 858), ([858, 49], 48), ([48, 102], 49), ([49, 51], 102), ([102, 52], 51), ([51, 53], 52), ([52, 54], 53), ([53, 55], 54), ([54, 56], 55), ([55, 57], 56), ([56, 58],

In [61]:
##Initialising Weights

import numpy as np
embedding_dim=5  #Size if the word vector

#Input Weight matrix
W1=np.random.randn(vocab_size,embedding_dim)

#Output Weight matrix
W2=np.random.randn(embedding_dim,vocab_size)

learning_rate=0.01

In [62]:
##Softmax Function

def softmax(x):
  exp_x=np.exp(x-np.max(x))
  return exp_x/exp_x.sum()

In [63]:
##Training (Basic Gradient Descent)

epochs=1000

for epoch in range(epochs):
  loss=0

  for context, target in training_data:

    #Forward Pass:
    h=np.mean(W1[context],axis=0)
    u=np.dot(h,W2)
    y_pred=softmax(u)

    #Loss (cross-entropy)
    loss -= np.log(y_pred[target])

    #Backpropagation
    error=y_pred
    error[target] -=1

    #Gradient for W2
    dW2=np.outer(h,error)

    #Gradient for W1
    dW1=np.dot(W2,error)/len(context)

    #Update W2
    W2 -= learning_rate * dW2

    #Update W1 (only context words)
    for word in context:
      W1[word] -= learning_rate * dW1 # Corrected from dW2 to dW1

if epoch % 200 == 0:
  print(f"Epoch {epoch}, Loss:{loss:.4f}")

In [64]:
print("\nWord Embeddings:\n")
for word in vocab:
  print(word,"->",W1[word_to_index[word]])


Word Embeddings:

artificial -> [-1.80461927  0.51419022  1.6022103   2.7939099   4.09981843]
intelligence -> [-1.56785301  2.13025641 -1.95589433  1.32886829  2.1112094 ]
ai -> [-2.13364916  1.78610378  1.70655115  2.22599079  0.25227909]
capability -> [ 2.14514672 -2.36742875  1.64766099  5.58486314 -0.57483758]
computational -> [-0.16066541  6.22900668 -0.67068222 -4.28799547 -1.7973959 ]
system -> [-0.52696211 -0.74631878  3.0230078  -2.77125275  1.21278851]
perform -> [ 0.75045264 -1.28974865 -2.01364314 -2.88723398 -0.27474497]
task -> [ 1.59702265  1.0261701  -2.77711882 -3.75000419 -3.73670577]
typically -> [-0.60042413 -2.852659   -0.54486209 -4.23819044 -6.27442771]
associated -> [ 0.72075293 -2.81906338 -4.22066859  0.57213902 -0.80616946]
human -> [-4.87169076  1.75592729  7.15484853  1.13574592  0.87015432]
intelligence -> [-1.56785301  2.13025641 -1.95589433  1.32886829  2.1112094 ]
learning -> [-2.40461227  0.84467464  6.73614084  7.40398196  2.0693575 ]
reasoning -> [-

#**Skip-Gram**

In [65]:
##Generate Training Data (window size=1)
window_size=1
training_data=[]

for i in range(window_size,len(words)-window_size):
    target= word_to_index[words[i]]

    for j in range(-window_size,window_size+1):
        if j!=0:
            context=word_to_index[words[i+j]]
            training_data.append((target,context))
print(training_data)
 #Word to index mapping
word_to_index={word: i for i,word in enumerate(vocab)}
index_to_word={i: word for word,i in word_to_index.items()}

[(223, 766), (223, 672), (672, 223), (672, 229), (229, 672), (229, 850), (850, 229), (850, 233), (233, 850), (233, 6), (6, 233), (6, 669), (669, 6), (669, 863), (863, 669), (863, 9), (9, 863), (9, 889), (889, 9), (889, 223), (223, 889), (223, 869), (869, 223), (869, 399), (399, 869), (399, 838), (838, 399), (838, 276), (276, 838), (276, 941), (941, 276), (941, 658), (658, 941), (658, 543), (543, 658), (543, 928), (928, 543), (928, 297), (297, 928), (297, 325), (325, 297), (325, 22), (22, 325), (22, 922), (922, 22), (922, 24), (24, 922), (24, 25), (25, 24), (25, 664), (664, 25), (664, 263), (263, 664), (263, 28), (28, 263), (28, 29), (29, 28), (29, 903), (903, 29), (903, 31), (31, 903), (31, 32), (32, 31), (32, 906), (906, 32), (906, 869), (869, 906), (869, 223), (223, 869), (223, 608), (608, 223), (608, 633), (633, 608), (633, 38), (38, 633), (38, 39), (39, 38), (39, 40), (40, 39), (40, 41), (41, 40), (41, 475), (475, 41), (475, 43), (43, 475), (43, 44), (44, 43), (44, 901), (901, 44),

In [66]:
##Initialising Weights

embedding_dim=5  #Size of the word vector

#Input Weight matrix
W1=np.random.randn(vocab_size,embedding_dim)

#Output Weight matrix
W2=np.random.randn(embedding_dim,vocab_size)

learning_rate=0.01

In [67]:
##Softmax Function

def softmax(x):
  exp_x=np.exp(x-np.max(x))
  return exp_x/exp_x.sum()

In [68]:
##Training (Basic Gradient Descent)

epochs=1000

for epoch in range(epochs):
  loss=0

  for target, context in training_data:

    #Forward Pass:
    h=W1[target]     #Target word vector
    u=np.dot(h,W2)
    y_pred=softmax(u)

    #Loss (cross-entropy)
    loss -= np.log(y_pred[context])

    #Backpropagation
    error=y_pred
    error[context] -=1

    #Gradient for W2
    dW2=np.outer(h,error)

    #Gradient for W1
    dW1=np.dot(W2,error)

    #Update Weights
    W2 -= learning_rate * dW2
    W1[target] -= learning_rate * dW1

if epoch % 200 == 0:
  print(f"Epoch {epoch}, Loss:{loss:.4f}")

In [69]:
##Word Embeddings

print("\nWord Embeddings:\n")
for word in vocab:
  print(word,"->",W1[word_to_index[word]])


Word Embeddings:

artificial -> [-0.38772702 -1.38399262  4.16208656 -1.94411236 -0.37836064]
intelligence -> [-0.37798867 -0.05149871  1.21433508  0.32692222 -0.16195686]
ai -> [ 0.16226065  0.12792288 -0.26230102 -0.20281137  0.14690858]
capability -> [-0.78590219  1.18288866  1.35287132 -2.12859339 -0.78608129]
computational -> [ 8.38681847e-04  1.60691428e-01  5.52231291e-01  1.13719068e-02
 -1.90992209e+00]
system -> [-0.93520271  1.95115897  0.11341039 -0.95471387  2.79985289]
perform -> [ 3.72253856 -0.55097891 -0.30483016 -0.68855218 -4.48144662]
task -> [-3.52830753  2.36404278 -0.1875783  -2.96146702  4.04491898]
typically -> [1.00661866 0.15797092 0.17941011 0.51970186 0.19378979]
associated -> [-2.60611363  0.99310579  1.70444567 -4.28380948  2.219052  ]
human -> [-0.11611985 -0.05881396  0.64588413  0.19126574 -0.04624357]
intelligence -> [-0.37798867 -0.05149871  1.21433508  0.32692222 -0.16195686]
learning -> [ 0.01409143 -0.07045568  0.8552502  -0.4869151   0.40595552]

#**Fast_Text**

In [70]:
pip install gensim

In [71]:
from gensim.models import FastText

In [72]:
#Train FastText model
model=FastText(
    clean_text,
    vector_size=50,   #size of word vectors
    window=3,         #context window size
    min_count=1,      #include all words
    epochs=100
)

In [73]:
#Get Word vector

vector=model.wv["learning"]
print(vector)

[-1.0521135e-03 -2.9445016e-05 -2.6681324e-04 -2.0629806e-03
 -1.5533755e-03  1.5935432e-03  2.3113489e-03 -8.0846180e-04
 -3.6553531e-03  1.6169866e-03  2.6812166e-04 -1.2354901e-03
  2.0685391e-05  1.8805226e-03  1.1954642e-03 -1.3146290e-03
  1.6609058e-03 -8.6206105e-04 -3.7580152e-04  3.7063458e-03
  1.4002168e-03 -1.0286489e-03 -3.4085761e-03  2.4692728e-03
  1.8361167e-03 -2.4178838e-03 -2.5847720e-03  2.7235423e-03
 -1.2194831e-03  5.0885845e-03  3.2921617e-03  2.9445691e-03
 -2.5521258e-03  1.1273614e-03 -8.8686880e-04 -1.2647441e-03
  9.6110343e-05  1.0053181e-03  4.1550426e-03  4.9981056e-03
  3.1284827e-03  2.3335047e-04  3.5504177e-03  2.7294530e-04
 -1.5858869e-03 -1.0548470e-03 -3.9235768e-03 -4.2642830e-03
 -2.5969148e-03  9.5642585e-04]


In [74]:
##OOV (out-of-vocublary) Example
print(model.wv["learner"])

[-7.6711811e-05 -5.4037623e-04  3.3403951e-04 -4.8356906e-06
  3.7976594e-03  2.5283645e-03  3.5243819e-04 -1.3900857e-03
 -6.1430186e-03  1.8944781e-03 -1.5276291e-03 -5.3535163e-04
  3.0906359e-03  4.6976621e-04  8.6582266e-04 -2.6877245e-04
 -2.9402555e-03 -1.3090279e-03 -9.5639843e-04  2.9214115e-03
 -4.7402810e-03  5.6520524e-04 -8.0471224e-04  3.6248676e-03
  1.8917612e-03 -4.3374496e-03 -4.8812554e-04  9.1552251e-04
  4.8038687e-04 -2.5842718e-03  3.0733773e-03 -1.2919354e-03
 -3.3281019e-03  8.1898768e-05 -1.9635416e-03  2.3626185e-03
  3.0756241e-03  1.4233589e-03 -2.3639975e-03 -1.0647179e-03
  7.2712876e-04 -8.9653110e-04  2.1852297e-03  2.4077191e-03
 -5.5985688e-04  6.5486779e-04 -4.1623679e-03 -2.9172162e-03
 -1.7306617e-03  4.2103506e-03]


In [75]:
#Word similarity

similarity=model.wv.similarity("machine","deep")
print("similarity:",similarity)

similarity: 0.050208047


#**Glove**

In [76]:
pip install gensim numpy

In [77]:
 #Load Pretrained GloVe model

import gensim.downloader as api
import numpy as np
from numpy.linalg import norm

In [78]:
#Load 50-dimensional GloVe vectors
glove=api.load("glove-wiki-gigaword-50")

### Accessing Word Vectors from GloVe Model

In [79]:
# Get word vector for 'learning'
learning_vector = glove['learning']
print("Vector for 'learning':", learning_vector)
print("Vector shape:", learning_vector.shape)

Vector for 'learning': [ 0.20461   0.48659  -0.55308  -0.27019   0.26336   0.15751  -0.28994
 -0.51824   0.051829  0.36225   0.37077   0.1322   -0.061377 -0.53606
 -0.34733  -0.043981 -0.086744  0.78305   0.41422   0.027996  0.23433
  0.98844  -0.41049   0.6206    1.3966   -0.65427  -0.18221  -1.0293
 -0.014741 -0.25384   3.227     0.39509  -0.33042  -1.229     0.29048
  0.33654  -0.24817   0.47105   0.32964   0.23997   0.088302 -0.91779
 -0.36671   0.9926    0.2185   -0.316     1.203     0.2699   -0.14093
  0.70785 ]
Vector shape: (50,)


### Word Similarity using GloVe

In [80]:
# Calculate similarity between 'machine' and 'deep'
similarity_glove = glove.similarity('machine', 'deep')
print("Similarity between 'machine' and 'deep':", similarity_glove)

Similarity between 'machine' and 'deep': 0.3714865
